# Proyecto Minería 4.0 – Fase 2 corregida

## Analítica de muestreo aurífero – Zona ALPACAY

Notebook reproducible para **Jupyter, Google Colab y un entorno Python de Antigravity**.

### Corrección metodológica

- Se trabaja con **una sola zona analítica: ALPACAY**.
- Las etiquetas originales `ALPACAY` y `AVISPA` se conservan únicamente como trazabilidad.
- La veta, labor y nivel se mantienen como variables descriptivas.
- Un tramo espacial alejado no se convierte automáticamente en otra zona.
- Los registros `BLANCO` y `DUPLICADO` permanecen para QA/QC, pero no ingresan como muestras de entrenamiento.
- `TRAZA` no se sustituye por un valor numérico sin conocer el límite de detección del laboratorio.

> Alcance: análisis exploratorio y preparación de datos. No constituye estimación de recursos ni reservas.

### Uso en Google Colab

1. Ejecuta las celdas en orden.
2. Cuando se solicite, carga el Excel de la Fase 1 o el modelo Excel de Power BI.
3. Al finalizar se descargará un ZIP con el reporte, las tablas y las cinco figuras.


In [ ]:
from pathlib import Path
import base64
import json
import shutil
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from scipy.spatial import cKDTree
from scipy.spatial.distance import cdist
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA
try:
    from IPython.display import display, HTML
except ImportError:
    def display(obj):
        print(obj)
    class HTML(str):
        pass

try:
    from google.colab import files
    EN_COLAB = True
except ImportError:
    files = None
    EN_COLAB = False

plt.style.use("seaborn-v0_8-whitegrid")
AZUL = "#4472C4"
NAVY = "#17365D"
ORO = "#D6A100"
ROJO = "#C00000"
VERDE = "#008C7A"

print("Entorno:", "Google Colab" if EN_COLAB else "Jupyter/Antigravity/local")


## 1. Carga del archivo Excel

El notebook acepta cualquiera de estas fuentes:

- `FASE2_MODELO_POWERBI_INTERNO_NO_PUBLICAR.xlsx`
- El libro de Fase 1 que contiene `DATA_LIMPIA`
- La base original que contiene `DATA_ORIGINAL`

En Colab aparecerá el selector de archivos. En un entorno local, coloca el Excel en la misma carpeta del notebook o en una subcarpeta llamada `data`.


In [ ]:
def seleccionar_excel():
    if EN_COLAB:
        cargados = files.upload()
        excels = [Path(nombre) for nombre in cargados if nombre.lower().endswith((".xlsx", ".xls"))]
        if not excels:
            raise FileNotFoundError("No se cargó ningún archivo Excel.")
        return excels[0]

    patrones = [Path.cwd().glob("*.xlsx"), (Path.cwd() / "data").glob("*.xlsx")]
    candidatos = [p for grupo in patrones for p in grupo if not p.name.startswith("~$")]
    if not candidatos:
        raise FileNotFoundError("Coloca el Excel junto al notebook o dentro de la carpeta data.")
    preferidos = [p for p in candidatos if "FASE2_MODELO_POWERBI" in p.name.upper()]
    return preferidos[0] if preferidos else candidatos[0]

ARCHIVO_EXCEL = seleccionar_excel()
print("Archivo seleccionado:", ARCHIVO_EXCEL.name)


## 2. Lectura y estandarización

La fuente original no se sobrescribe. Se crea una tabla analítica con nombres de campos homogéneos y una zona única.


In [ ]:
def texto_limpio(serie):
    return serie.astype("string").str.strip().replace({"": pd.NA, "NAN": pd.NA, "<NA>": pd.NA})


def cargar_datos(ruta):
    libro = pd.ExcelFile(ruta)

    if "Fact_Muestras" in libro.sheet_names:
        df = pd.read_excel(ruta, sheet_name="Fact_Muestras")
        dimensiones = [
            ("Dim_Zona", "ZonaKey"),
            ("Dim_Tipo", "TipoKey"),
            ("Dim_Labor", "LaborKey"),
            ("Dim_Grupo", "GrupoKey"),
        ]
        for hoja, llave in dimensiones:
            if hoja in libro.sheet_names and llave in df.columns:
                dim = pd.read_excel(ruta, sheet_name=hoja)
                columnas_nuevas = [c for c in dim.columns if c == llave or c not in df.columns]
                df = df.merge(dim[columnas_nuevas], on=llave, how="left")

        if "FechaKey" in df.columns:
            df["Fecha"] = pd.to_datetime(
                df["FechaKey"].astype("Int64").astype("string"),
                format="%Y%m%d",
                errors="coerce",
            )
        fuente = "Modelo Power BI – Fact_Muestras"

    else:
        hoja = "DATA_LIMPIA" if "DATA_LIMPIA" in libro.sheet_names else "DATA_ORIGINAL"
        df = pd.read_excel(ruta, sheet_name=hoja)
        df.columns = [str(c).strip() for c in df.columns]
        renombrar = {
            "FECHA": "Fecha", "CODIGO": "Codigo", "ZONA": "Zona", "VETA": "Veta",
            "TIPO": "Tipo", "LABOR": "Labor", "GRUPO": "Grupo", "Nv.": "Nivel",
            "POT (m)": "Potencia_m", "Au (gr/TM)": "Au_Reporte",
            "ESTE": "Este", "NORTE": "Norte", "COTA": "Cota",
            "FLAG_COORD": "FlagCoord", "FLAG_AU": "FlagAu", "FLAG_TIPO": "FlagTipo",
            "INCLUIR_ML": "IncluirML", "MOTIVO_EXCLUSION": "MotivoExclusion",
        }
        df = df.rename(columns=renombrar)
        df["Fecha"] = pd.to_datetime(df.get("Fecha"), errors="coerce")
        fuente = f"Libro de Fase 1 – {hoja}"

    if "Codigo" not in df.columns:
        df["Codigo"] = [f"M-{i+1:05d}" for i in range(len(df))]
    if "MuestraKey" not in df.columns:
        df["MuestraKey"] = np.arange(1, len(df) + 1)

    for columna in ["Zona", "Veta", "Tipo", "Labor", "Grupo", "Codigo"]:
        if columna not in df.columns:
            df[columna] = pd.NA
        df[columna] = texto_limpio(df[columna])

    if "Nivel" not in df.columns:
        df["Nivel"] = np.nan
    if "Potencia_m" not in df.columns:
        df["Potencia_m"] = np.nan
    for columna in ["Este", "Norte", "Cota", "Nivel", "Potencia_m"]:
        df[columna] = pd.to_numeric(df[columna], errors="coerce")

    if "Au_g_t" not in df.columns:
        if "Au_Reporte" not in df.columns:
            df["Au_Reporte"] = pd.NA
        df["Au_g_t"] = pd.to_numeric(df["Au_Reporte"], errors="coerce")
    else:
        df["Au_g_t"] = pd.to_numeric(df["Au_g_t"], errors="coerce")
        if "Au_Reporte" not in df.columns:
            df["Au_Reporte"] = df["Au_g_t"]

    coord_ok = df[["Este", "Norte", "Cota"]].notna().all(axis=1)
    tipo_norm = df["Tipo"].fillna("").str.upper()
    es_qaqc = tipo_norm.str.contains("BLANCO|DUPLICADO", regex=True)

    if "FlagCoord" not in df.columns:
        df["FlagCoord"] = np.where(coord_ok, "OK", "INCOMPLETA")
    if "FlagAu" not in df.columns:
        traza = df["Au_Reporte"].astype("string").str.upper().str.contains("TRAZA", na=False)
        df["FlagAu"] = np.select([df["Au_g_t"].notna(), traza], ["OK", "TRAZA"], default="VACIO")
    if "IncluirML" not in df.columns:
        df["IncluirML"] = np.where(coord_ok & df["Au_g_t"].notna() & ~es_qaqc, "SI", "NO")

    # Corrección principal: una sola zona analítica.
    df["Zona_original"] = df["ZonaOriginal"] if "ZonaOriginal" in df.columns else df["Zona"]
    df["Zona_analitica"] = "ALPACAY"
    return df, fuente


datos, FUENTE = cargar_datos(ARCHIVO_EXCEL)
print(FUENTE)
print(f"Registros cargados: {len(datos):,}")
display(datos[["MuestraKey", "Zona_original", "Zona_analitica", "Veta", "Labor", "Nivel", "Au_g_t"]].head())


## 3. QA/QC y población candidata

Se mantienen todos los registros. La población candidata para análisis/modelamiento exige coordenadas completas, Au numérico y exclusión de controles QA/QC.


In [ ]:
coord_ok = datos[["Este", "Norte", "Cota"]].notna().all(axis=1)
tipo_norm = datos["Tipo"].fillna("").str.upper()
es_qaqc = tipo_norm.str.contains("BLANCO|DUPLICADO", regex=True)
es_traza = (
    datos["Au_Reporte"].astype("string").str.upper().str.contains("TRAZA", na=False)
    | datos["FlagAu"].astype("string").str.upper().eq("TRAZA")
)
candidatos = datos.loc[datos["IncluirML"].astype("string").str.upper().eq("SI") & datos["Au_g_t"].notna()].copy()

q1 = candidatos["Au_g_t"].quantile(0.25)
q3 = candidatos["Au_g_t"].quantile(0.75)
umbral_iqr = q3 + 1.5 * (q3 - q1)

metricas = {
    "Total de registros": len(datos),
    "Coordenadas completas": int(coord_ok.sum()),
    "Cobertura espacial (%)": 100 * coord_ok.mean(),
    "Au numérico": int(datos["Au_g_t"].notna().sum()),
    "Au reportado como TRAZA": int(es_traza.sum()),
    "Registros QA/QC": int(es_qaqc.sum()),
    "Candidatos para ML": len(candidatos),
    "Au mínimo (g/t)": candidatos["Au_g_t"].min(),
    "Au mediana (g/t)": candidatos["Au_g_t"].median(),
    "Au promedio (g/t)": candidatos["Au_g_t"].mean(),
    "Au P95 (g/t)": candidatos["Au_g_t"].quantile(0.95),
    "Au máximo (g/t)": candidatos["Au_g_t"].max(),
    "Umbral IQR superior (g/t)": umbral_iqr,
    "Registros sobre umbral IQR": int((candidatos["Au_g_t"] > umbral_iqr).sum()),
}

tabla_metricas = pd.DataFrame({"Indicador": metricas.keys(), "Resultado": metricas.values()})
display(tabla_metricas.round({"Resultado": 3}))


## 4. Validación espacial de una sola zona

La lógica no utiliza `Zona_original` para crear grupos. Se estudia la continuidad geométrica con:

- distancia al vecino más cercano;
- componentes conectados con una tolerancia de 20 m;
- eje principal de las coordenadas;
- identificación de tramos alejados dentro de ALPACAY.

La tolerancia de 20 m es un parámetro exploratorio y puede cambiarse después de revisar todos los planos CAD/DXF.


In [ ]:
EPS_ESPACIAL_M = 20.0
espacial = datos.loc[coord_ok].copy()
puntos = espacial[["Este", "Norte"]].drop_duplicates().reset_index(drop=True)
xy = puntos[["Este", "Norte"]].to_numpy()

arbol = cKDTree(xy)
distancias, _ = arbol.query(xy, k=2)
puntos["Distancia_vecino_m"] = distancias[:, 1]

etiquetas = DBSCAN(eps=EPS_ESPACIAL_M, min_samples=1).fit_predict(xy)
puntos["Componente"] = etiquetas
componente_principal = puntos["Componente"].value_counts().idxmax()
puntos["Tramo_espacial"] = np.where(
    puntos["Componente"].eq(componente_principal), "Principal", "Alejado"
)

espacial = espacial.merge(puntos, on=["Este", "Norte"], how="left")
datos = datos.merge(
    puntos[["Este", "Norte", "Distancia_vecino_m", "Tramo_espacial"]],
    on=["Este", "Norte"], how="left"
)
candidatos = candidatos.merge(
    puntos[["Este", "Norte", "Distancia_vecino_m", "Tramo_espacial"]],
    on=["Este", "Norte"], how="left"
)

pca = PCA(n_components=2).fit(xy)
proyectado = pca.transform(xy)
extension_eje_m = float(np.ptp(proyectado[:, 0]))
extension_transversal_m = float(np.ptp(proyectado[:, 1]))
azimut_eje = float((np.degrees(np.arctan2(pca.components_[0, 0], pca.components_[0, 1])) + 360) % 180)

principales = puntos.loc[puntos["Tramo_espacial"].eq("Principal"), ["Este", "Norte"]].to_numpy()
alejados = puntos.loc[puntos["Tramo_espacial"].eq("Alejado"), ["Este", "Norte"]].to_numpy()
separacion_m = float(cdist(principales, alejados).min()) if len(alejados) else 0.0

resumen_espacial = pd.DataFrame({
    "Indicador": [
        "Puntos XY únicos", "Distancia mediana al vecino (m)",
        "Distancia máxima al vecino (m)", "Extensión sobre el eje principal (m)",
        "Extensión transversal (m)", "Variación explicada por el eje principal (%)",
        "Azimut aproximado del eje (°)", "Separación mínima del tramo alejado (m)",
        "Registros en tramo alejado",
    ],
    "Resultado": [
        len(puntos), np.median(puntos["Distancia_vecino_m"]), puntos["Distancia_vecino_m"].max(),
        extension_eje_m, extension_transversal_m, 100 * pca.explained_variance_ratio_[0],
        azimut_eje, separacion_m, int(espacial["Tramo_espacial"].eq("Alejado").sum()),
    ],
})
display(resumen_espacial.round({"Resultado": 3}))

if len(alejados):
    print("Labores del tramo alejado:")
    display(
        espacial.loc[espacial["Tramo_espacial"].eq("Alejado")]
        .groupby(["Labor", "Nivel"], dropna=False)
        .size().rename("Registros").reset_index().sort_values("Registros", ascending=False)
    )

print("Conclusión: los nombres originales no se usarán para separar zonas. Todo pertenece a ALPACAY.")


## 5. Visualizaciones

Las coordenadas se muestran de forma relativa para evitar publicar la ubicación UTM exacta.


In [ ]:
RESULTADOS = Path("resultados_fase2_alpacay")
FIGURAS = RESULTADOS / "figuras"
FIGURAS.mkdir(parents=True, exist_ok=True)

figuras = []

def guardar_figura(nombre):
    ruta = FIGURAS / nombre
    plt.tight_layout()
    plt.savefig(ruta, dpi=170, bbox_inches="tight")
    plt.show()
    plt.close()
    figuras.append(ruta)
    return ruta


In [ ]:
# Figura 1: distribución de Au
plt.figure(figsize=(10, 5))
positivos = candidatos.loc[candidatos["Au_g_t"] > 0, "Au_g_t"]
bins = np.geomspace(positivos.min(), positivos.max(), 25)
plt.hist(positivos, bins=bins, color=AZUL, edgecolor="white")
plt.axvline(positivos.median(), color=ORO, linewidth=2, label=f"Mediana: {positivos.median():.3f} g/t")
plt.xscale("log")
plt.xlabel("Au (g/t) – escala logarítmica")
plt.ylabel("Número de muestras")
plt.title("Distribución de leyes de Au – ALPACAY")
plt.legend()
guardar_figura("01_distribucion_au.png")


In [ ]:
# Figura 2: mapa espacial relativo y tramo alejado
mapa = candidatos.dropna(subset=["Este", "Norte", "Au_g_t"]).copy()
mapa["Este_rel_m"] = mapa["Este"] - espacial["Este"].min()
mapa["Norte_rel_m"] = mapa["Norte"] - espacial["Norte"].min()

plt.figure(figsize=(12, 6))
for tramo, borde, marcador in [("Principal", "none", "o"), ("Alejado", ROJO, "s")]:
    g = mapa.loc[mapa["Tramo_espacial"].fillna("Principal").eq(tramo)]
    if len(g):
        plt.scatter(
            g["Este_rel_m"], g["Norte_rel_m"], c=g["Au_g_t"], cmap="viridis",
            norm=LogNorm(vmin=mapa["Au_g_t"].min(), vmax=mapa["Au_g_t"].max()),
            s=42, marker=marcador, edgecolors=borde, linewidths=1.0,
            label=f"Tramo {tramo.lower()}",
        )
plt.colorbar(label="Au (g/t)")
plt.xlabel("Este relativo (m)")
plt.ylabel("Norte relativo (m)")
plt.title("Distribución espacial relativa – una sola zona ALPACAY")
plt.legend()
guardar_figura("02_mapa_espacial_relativo.png")


In [ ]:
# Figura 3: Au por labor, reemplaza la comparación incorrecta entre zonas
top_labores = candidatos["Labor"].value_counts().head(10).index
por_labor = candidatos.loc[candidatos["Labor"].isin(top_labores)].copy()
orden = por_labor.groupby("Labor")["Au_g_t"].median().sort_values().index

plt.figure(figsize=(11, 6))
plt.boxplot(
    [por_labor.loc[por_labor["Labor"].eq(labor), "Au_g_t"].clip(lower=1e-4) for labor in orden],
    tick_labels=orden, showfliers=False, patch_artist=True,
    boxprops={"facecolor": AZUL, "alpha": 0.65}, medianprops={"color": ORO, "linewidth": 2},
)
plt.yscale("log")
plt.xlabel("Labor")
plt.ylabel("Au (g/t) – escala logarítmica")
plt.title("Distribución de Au por labor – ALPACAY")
plt.xticks(rotation=35, ha="right")
guardar_figura("03_au_por_labor.png")


In [ ]:
# Figura 4: relación entre potencia y Au
relacion = candidatos.dropna(subset=["Potencia_m", "Au_g_t"]).copy()
rho = relacion[["Potencia_m", "Au_g_t"]].corr(method="spearman").iloc[0, 1]

plt.figure(figsize=(10, 6))
plt.scatter(relacion["Potencia_m"], relacion["Au_g_t"], alpha=0.65, color=VERDE, edgecolors="white", linewidths=0.3)
plt.yscale("log")
plt.xlabel("Potencia (m)")
plt.ylabel("Au (g/t) – escala logarítmica")
plt.title(f"Potencia frente a Au – Spearman ρ = {rho:.3f}")
guardar_figura("04_potencia_vs_au.png")


In [ ]:
# Figura 5: tendencia mensual
temporal = candidatos.dropna(subset=["Fecha"]).copy()
temporal["Mes"] = temporal["Fecha"].dt.to_period("M").dt.to_timestamp()
mensual = temporal.groupby("Mes").agg(
    Muestras=("MuestraKey", "size"),
    Au_mediana=("Au_g_t", "median"),
    Au_promedio=("Au_g_t", "mean"),
).reset_index()

fig, ax1 = plt.subplots(figsize=(12, 6))
ax1.plot(mensual["Mes"], mensual["Au_mediana"], color=AZUL, marker="o", label="Mediana Au")
ax1.set_ylabel("Au mediana (g/t)", color=AZUL)
ax1.set_xlabel("Mes")
ax2 = ax1.twinx()
ax2.bar(mensual["Mes"], mensual["Muestras"], width=20, color=ORO, alpha=0.25, label="Muestras")
ax2.set_ylabel("Número de muestras", color="#7F6000")
plt.title("Tendencia mensual de Au y cantidad de muestras – ALPACAY")
guardar_figura("05_tendencia_mensual.png")


## 6. Tablas de salida y reporte HTML

El reporte utiliza coordenadas relativas y contiene las imágenes incrustadas. Puede abrirse sin conservar una carpeta externa de figuras.


In [ ]:
resumen_labor = (
    candidatos.groupby("Labor", dropna=False)
    .agg(
        Muestras=("MuestraKey", "size"),
        Au_media=("Au_g_t", "mean"),
        Au_mediana=("Au_g_t", "median"),
        Au_p95=("Au_g_t", lambda s: s.quantile(0.95)),
        Au_max=("Au_g_t", "max"),
        Potencia_media=("Potencia_m", "mean"),
    )
    .reset_index().sort_values("Muestras", ascending=False)
)

salida = candidatos.copy().reset_index(drop=True)
salida["MuestraAnonima"] = [f"M-{i+1:04d}" for i in range(len(salida))]
salida["Este_rel_m"] = salida["Este"] - espacial["Este"].min()
salida["Norte_rel_m"] = salida["Norte"] - espacial["Norte"].min()
columnas_publicables = [
    "MuestraAnonima", "Fecha", "Zona_analitica", "Veta", "Labor", "Nivel", "Tipo",
    "Potencia_m", "Au_g_t", "Este_rel_m", "Norte_rel_m", "Cota", "Tramo_espacial",
]
salida[columnas_publicables].to_csv(RESULTADOS / "muestras_candidatas_anonimizadas.csv", index=False)
tabla_metricas.to_csv(RESULTADOS / "resumen_metricas.csv", index=False)
resumen_espacial.to_csv(RESULTADOS / "resumen_espacial.csv", index=False)
resumen_labor.to_csv(RESULTADOS / "resumen_por_labor.csv", index=False)

display(resumen_labor.head(12).round({
    "Au_media": 3, "Au_mediana": 3, "Au_p95": 3,
    "Au_max": 3, "Potencia_media": 3,
}))


In [ ]:
def imagen_embebida(ruta):
    codificada = base64.b64encode(Path(ruta).read_bytes()).decode("ascii")
    return f'<section><img src="data:image/png;base64,{codificada}" alt="{Path(ruta).stem}"></section>'

filas_metricas = "".join(
    f"<tr><td>{k}</td><td>{v:.3f}</td></tr>" if isinstance(v, (float, np.floating))
    else f"<tr><td>{k}</td><td>{v}</td></tr>"
    for k, v in metricas.items()
)
filas_espaciales = "".join(
    f"<tr><td>{fila.Indicador}</td><td>{fila.Resultado:.3f}</td></tr>"
    for fila in resumen_espacial.itertuples(index=False)
)
imagenes_html = "".join(imagen_embebida(ruta) for ruta in figuras)

reporte_html = f'''<!doctype html>
<html lang="es"><head><meta charset="utf-8"><title>Fase 2 corregida – ALPACAY</title>
<style>
body{{font-family:Arial,sans-serif;max-width:1100px;margin:40px auto;color:#1f2937;line-height:1.45}}
h1,h2{{color:{NAVY}}}.nota{{background:#fff2cc;border-left:5px solid {ORO};padding:12px}}
table{{border-collapse:collapse;width:100%;margin:18px 0}}th{{background:{NAVY};color:white}}
td,th{{padding:8px;border-bottom:1px solid #d9e2f3;text-align:left}}
img{{width:100%;max-width:1000px;border:1px solid #e5e7eb;margin:12px 0}}
footer{{margin-top:30px;color:#6b7280;font-size:.9rem}}
</style></head><body>
<h1>Proyecto Minería 4.0 – Fase 2 corregida</h1>
<p>Análisis exploratorio del muestreo aurífero de la única zona analítica <strong>ALPACAY</strong>.</p>
<div class="nota"><strong>Criterio corregido:</strong> AVISPA se conserva solamente como etiqueta original. No se usa como zona separada. Los tramos alejados permanecen dentro de ALPACAY.</div>
<h2>Indicadores de calidad y Au</h2><table><tr><th>Indicador</th><th>Resultado</th></tr>{filas_metricas}</table>
<h2>Validación espacial</h2><table><tr><th>Indicador</th><th>Resultado</th></tr>{filas_espaciales}</table>
<h2>Resumen por labor</h2>{resumen_labor.to_html(index=False, float_format=lambda x: f"{x:.3f}")}
<h2>Visualizaciones</h2>{imagenes_html}
<footer>Reporte autónomo. Las coordenadas horizontales se presentan de forma relativa. Este análisis no constituye estimación de recursos o reservas.</footer>
</body></html>'''

RUTA_REPORTE = RESULTADOS / "REPORTE_FASE2_ALPACAY_CORREGIDO.html"
RUTA_REPORTE.write_text(reporte_html, encoding="utf-8")
print("Reporte generado:", RUTA_REPORTE)
display(HTML('<p><strong>Reporte HTML generado correctamente con cinco imágenes incrustadas.</strong></p>'))


## 7. Paquete de resultados

El ZIP contiene el reporte HTML autónomo, las tablas CSV y las figuras PNG. En Google Colab se iniciará su descarga.


In [ ]:
RUTA_ZIP = Path("FASE2_ALPACAY_RESULTADOS_COLAB.zip")
with zipfile.ZipFile(RUTA_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for archivo in RESULTADOS.rglob("*"):
        if archivo.is_file():
            zf.write(archivo, archivo.relative_to(RESULTADOS.parent))

print("Paquete generado:", RUTA_ZIP)
print("Archivos incluidos:")
with zipfile.ZipFile(RUTA_ZIP) as zf:
    for nombre in zf.namelist():
        print(" -", nombre)

if EN_COLAB:
    files.download(str(RUTA_ZIP))


## Interpretación para continuar el proyecto

- La variable analítica de zona queda fija en **ALPACAY**.
- `Zona_original` permanece únicamente para auditar el origen de los datos.
- El análisis por labor sustituye la comparación incorrecta entre zonas.
- Los tramos alejados se identifican con geometría, pero no se excluyen ni se renombran como otra zona.
- Antes de entrenar Machine Learning se deberá aplicar validación espacial para evitar que muestras vecinas aparezcan simultáneamente en entrenamiento y prueba.
- La superposición final con labores requiere los planos en DXF o una fuente CAD que pueda leerse programáticamente.
